In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded")

ModuleNotFoundError: No module named 'sklearn'

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded")

ModuleNotFoundError: No module named 'sklearn'

In [3]:
import sys
!{sys.executable} -m pip install scikit-learn xgboost imbalanced-learn joblib


  Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
  Using cached imbalanced_learn-0.14.2-py3-none-any.whl.metadata (8.9 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached sklearn_compat-0.1.6-py3-none-any.whl.metadata (22 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl (8.3 MB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)
Using cached imbalanced_learn-0.14.2-py3-none-any.whl (236 kB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl (37.4 MB)
Using cached sklearn_compat-0.1.6-py3-none-any.whl (22 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Using cached clou


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded")

Libraries loaded


In [5]:
df = pd.read_csv('../data/creditcard.csv')

X = df.drop('Class', axis=1)
y = df['Class']

# Scale the Amount column
scaler = StandardScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])
X['Time'] = scaler.fit_transform(X[['Time']])

# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} transactions")
print(f"Test set:     {X_test.shape[0]:,} transactions")
print(f"\nFraud in training: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Fraud in test:     {y_test.sum()} ({y_test.mean()*100:.2f}%)")

Training set: 227,845 transactions
Test set:     56,962 transactions

Fraud in training: 394 (0.17%)
Fraud in test:     98 (0.17%)


In [6]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.value_counts()[0]:,} legitimate | {y_train.value_counts()[1]:,} fraud")
print(f"After SMOTE:  {y_train_balanced.value_counts()[0]:,} legitimate | {y_train_balanced.value_counts()[1]:,} fraud")
print(f"\nNew training size: {len(X_train_balanced):,} transactions")

Before SMOTE: 227,451 legitimate | 394 fraud
After SMOTE:  227,451 legitimate | 227,451 fraud

New training size: 454,902 transactions


In [7]:
# Train XGBoost model
print("Training model... ⏳")

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=1,  # 1 because SMOTE already balanced the data
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

model.fit(X_train_balanced, y_train_balanced)

print("Model trained! ✅")

Training model... ⏳
Model trained! ✅


In [8]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("=" * 50)
print("MODEL EVALUATION")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

MODEL EVALUATION
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     56864
       Fraud       0.35      0.88      0.50        98

    accuracy                           1.00     56962
   macro avg       0.68      0.94      0.75     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC Score: 0.9752
